# Homework 2 — Data cleaning and validation
**Prabhutosh Kumar**

Run from this submission directory. The source database is distributed through bCourses; it is never committed. We work on a local copy and leave the source tables unchanged.

`ts`: observation date; `ticker`: vendor equity label; `open/high/low/close`: raw daily prices; `adj_close`: vendor adjusted close; `volume`: traded share count. Raw OHLC bounds do not constrain adjusted close.

In [1]:
from pathlib import Path
import sqlite3
import shutil
import pandas as pd
from IPython.display import display
from analysis import normalize, diagnose, missing_dates, clean_data, save_cleaned

db_path = Path("data/data.db")
source_path = Path("../../../Lectures/Lecture 4/data/data.db")
if not db_path.exists():
    if not source_path.is_file():
        raise FileNotFoundError("Download data.db from bCourses to Lectures/Lecture 4/data/, then rerun.")
    db_path.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source_path, db_path)
with sqlite3.connect(db_path) as connection:
    raw = pd.read_sql_query("SELECT * FROM ohlc_hw", connection)
    reference = pd.read_sql_query("SELECT * FROM ohlc", connection)
# Accommodate the lecture reference date-column convention explicitly.
if "date" in reference.columns and "ts" not in reference.columns:
    reference = reference.rename(columns={"date": "ts"})
raw.info()
display(raw.describe(include="all"))
normalized = normalize(raw)
print("Date range:", normalized.ts.min(), "to", normalized.ts.max())
print("Tickers:", normalized.ticker.dropna().unique().tolist())
display(raw.isna().sum().rename("missing_cells").to_frame())
display(normalized.groupby(["original_ticker", "ticker"], dropna=False).size().rename("rows").reset_index())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 397711 entries, 0 to 397710
Data columns (total 8 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   ts         397711 non-null  object 
 1   open       397711 non-null  float64
 2   high       397711 non-null  float64
 3   low        397711 non-null  float64
 4   close      397711 non-null  float64
 5   adj_close  377856 non-null  float64
 6   volume     397711 non-null  int64  
 7   ticker     397711 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 24.3+ MB


,ts,open,high,low,close,adj_close,volume,ticker
count,397711,397711.000000,397711.000000,397711.000000,397711.000000,377856.000000,3.977110e+05,397711
unique,15868,NaN,NaN,NaN,NaN,NaN,NaN,150
top,2020-11-24,NaN,NaN,NaN,NaN,NaN,NaN,PG
freq,40,NaN,NaN,NaN,NaN,NaN,NaN,15450
mean,NaN,46.285479,46.747660,45.816279,46.292903,51.652250,3.097773e+07,NaN
std,NaN,67.132594,67.778210,66.477178,67.141879,194.584045,1.266237e+08,NaN
min,NaN,0.000000,0.005208,0.004801,0.005208,0.002945,0.000000e+00,NaN
25%,NaN,4.209938,4.240000,4.160000,4.200000,1.390000,1.730700e+06,NaN
50%,NaN,19.850000,20.060000,19.630000,19.860000,11.960000,4.648200e+06,NaN
75%,NaN,57.530000,58.130000,56.950000,57.530000,43.770000,1.102380e+07,NaN


Date range: 1962-01-02 00:00:00 to 2024-12-06 00:00:00
Tickers: ['AAPL', 'AMGN', 'AMZN', 'AXP', 'BA', 'CAT', 'CRM', 'CSCO', 'CVX', 'DIS', 'GS', 'HD', 'HON', 'IBM', 'JNJ', 'JPM', 'KO', 'MCD', 'MMM', 'MRK', 'MSFT', 'NKE', 'NVDA', 'PG', 'SHW', 'TRV', 'UNH', 'V', 'VZ', 'WMT']


,missing_cells
ts,0
open,0
high,0
low,0
close,0
adj_close,19855
volume,0
ticker,0


,original_ticker,ticker,rows
0,"<a target=""_blank""><span style=""color: blue"">A...",AAPL,307
1,"<a target=""_blank""><span style=""color: blue"">A...",AMGN,294
2,"<a target=""_blank""><span style=""color: blue"">A...",AMZN,185
3,"<a target=""_blank""><span style=""color: blue"">A...",AXP,356
4,"<a target=""_blank""><span style=""color: blue"">B...",BA,462
...,...,...,...
145,TRV,TRV,12860
146,UNH,UNH,9835
147,V,V,4086
148,VZ,VZ,9999


## Diagnosis and decisions
The full pre-repair report permits multiple issues per observation. High severity covers invalid keys, ambiguous duplicates, nonpositive/nonfinite values and impossible bounds. Missing cells, absent dates and statistical return flags are medium; zero volume is low because it can be legitimate.

Calendar gaps use the supplied `ohlc` dates **per ticker**, over that ticker's full supplied history; exchange holidays are therefore not invented as gaps. We report missing rows without manufacturing prices.

Exact duplicates are removed after label normalization. All members of conflicting keys are quarantined. All-raw-price-missing rows are removed before filling. Missing raw prices use the preceding validated observation within the same ticker; leading gaps and repairs that violate today's range are quarantined. Missing volume becomes zero with an unknown-volume flag.

A trusted, unambiguous matched reference row supplies a date-specific `adj_close/close` factor. We reconstruct adjusted close when missing or inconsistent, including after raw repairs. We never carry factors across dates or replace the dirty raw dataset with the reference. Without a factor, a repair that needs one is quarantined.

The retrospective two-tail 3σ screen uses simple raw-close returns and sample standard deviation. Undefined returns are excluded; zero/undefined standard deviations yield no flags. Extreme returns remain review flags: without confirmed evidence we cannot label them splits, news or errors. Impossible prices are quarantined rather than interpolated without evidence.

In [2]:
cleaned, before, after, removed, quarantine = clean_data(raw, reference)
display(before)
display(before.groupby(["issue_type", "severity"]).size().rename("issues").to_frame())
calendar = normalize(reference)
gaps = pd.DataFrame(missing_dates(normalized, calendar), columns=before.columns)
for ticker, history in calendar.groupby("ticker"):
    count = int(gaps.ticker.eq(ticker).sum()) if not gaps.empty else 0
    print(f"Ticker {ticker} is missing {count} trading days between {history.ts.min().date()} and {history.ts.max().date()}")
display(pd.DataFrame({"category": ["input", "retained", "removed", "quarantined"],
                      "rows": [len(raw), len(cleaned), len(removed), len(quarantine)]}))
display(removed)
display(quarantine)
display(cleaned.repair_flags.value_counts().rename("rows").to_frame())
display(after)
print("Post-repair gaps and return/zero-volume flags can remain; invalid retained prices cannot.")
display(pd.concat({"before": normalized.describe(), "after": cleaned.describe()}, axis=1))

,ticker,issue_type,issue_date,description,severity
0,AAPL,missing_value,1981-01-08,adj_close is missing,medium
1,AAPL,missing_value,1981-01-13,adj_close is missing,medium
2,AAPL,missing_value,1981-02-12,adj_close is missing,medium
3,AAPL,missing_value,1981-04-16,adj_close is missing,medium
4,AAPL,missing_value,1981-05-19,adj_close is missing,medium
...,...,...,...,...,...
84602,WMT,duplicate_key,2024-09-19,"Repeated normalized (ticker, ts); distinguish ...",high
84603,WMT,duplicate_key,2024-09-30,"Repeated normalized (ticker, ts); distinguish ...",high
84604,WMT,duplicate_key,2024-09-30,"Repeated normalized (ticker, ts); distinguish ...",high
84605,WMT,duplicate_key,2024-10-17,"Repeated normalized (ticker, ts); distinguish ...",high


,,issues
issue_type,severity,
duplicate_key,high,58920
extreme_return,medium,5741
invalid_value,high,5
missing_value,medium,19855
ohlc_bounds,high,5
zero_volume,low,81


Ticker AAPL is missing 0 trading days between 1980-12-12 and 2024-10-18
Ticker AMGN is missing 0 trading days between 1983-06-17 and 2024-10-18
Ticker AMZN is missing 0 trading days between 1997-05-15 and 2024-12-06
Ticker AXP is missing 0 trading days between 1972-04-03 and 2024-10-18
Ticker BA is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CAT is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker CRM is missing 0 trading days between 2004-06-23 and 2024-10-18
Ticker CSCO is missing 0 trading days between 1990-02-16 and 2024-10-18
Ticker CVX is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker DIS is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticker GS is missing 0 trading days between 1999-05-04 and 2024-10-18
Ticker HD is missing 0 trading days between 1981-09-22 and 2024-10-18
Ticker HON is missing 0 trading days between 1962-01-02 and 2024-10-18
Ticker IBM is missing 0 trading days between 1962-01-02 and 2024-12-06
Ticke

,category,rows
0,input,397711
1,retained,368246
2,removed,29460
3,quarantined,5


,ts,open,high,low,close,adj_close,volume,ticker,original_ticker,source_row,repair_flags,reason
0,1981-01-02,0.15402,0.15402,0.15402,0.15402,0.118730,21660820,AAPL,AAPL,14,,exact duplicate after normalization
1,1981-01-05,0.15067,0.15067,0.15067,0.15067,0.116150,35728032,AAPL,AAPL,16,,exact duplicate after normalization
2,1981-02-10,0.12165,0.12165,0.12165,0.12165,0.093779,18345616,AAPL,AAPL,43,,exact duplicate after normalization
3,1981-02-27,0.11830,0.11830,0.11830,0.11830,0.091197,14761612,AAPL,AAPL,56,,exact duplicate after normalization
4,1981-03-05,0.11551,0.11551,0.11551,0.11551,0.089046,5376004,AAPL,AAPL,61,,exact duplicate after normalization
...,...,...,...,...,...,...,...,...,...,...,...,...
29455,2024-08-23,75.60000,75.81000,75.01000,75.70000,75.700000,12299100,WMT,WMT,397667,,exact duplicate after normalization
29456,2024-09-18,78.64000,79.41000,78.28000,79.03000,NaN,15684900,WMT,"<a target=""_blank""><span style=""font-weight: b...",397685,,exact duplicate after normalization
29457,2024-09-19,79.05000,79.09000,77.49000,78.04000,78.040000,21525700,WMT,WMT,397687,,exact duplicate after normalization
29458,2024-09-30,79.89000,80.94000,79.86000,80.75000,80.750000,19008187,WMT,WMT,397695,,exact duplicate after normalization


,ts,open,high,low,close,adj_close,volume,ticker,original_ticker,source_row,repair_flags,reason
0,1962-01-02,0.0,3.30000,3.24000,3.30000,0.335750,105840,CVX,CVX,93926,,"unresolved raw bounds/price, volume, or date-s..."
1,1962-01-02,0.0,0.22338,0.22222,0.22338,0.064012,0,JNJ,JNJ,181061,,"unresolved raw bounds/price, volume, or date-s..."
2,1966-07-05,0.0,0.27366,0.26749,0.26955,0.118090,388800,MCD,MCD,230450,,"unresolved raw bounds/price, volume, or date-s..."
3,1962-01-02,0.0,0.39703,0.39317,0.39537,0.068447,633830,MRK,MRK,263321,,"unresolved raw bounds/price, volume, or date-s..."
4,1962-01-02,0.0,1.44000,1.42000,1.43000,0.259300,192000,PG,PG,309937,,"unresolved raw bounds/price, volume, or date-s..."


,rows
repair_flags,
,339455
adj_close_from_reference_date_factor,28791


,ticker,issue_type,issue_date,description,severity
0,CVX,missing_date,1962-01-02,Date present in supplied ohlc calendar,medium
1,JNJ,missing_date,1962-01-02,Date present in supplied ohlc calendar,medium
2,MCD,missing_date,1966-07-05,Date present in supplied ohlc calendar,medium
3,MRK,missing_date,1962-01-02,Date present in supplied ohlc calendar,medium
4,PG,missing_date,1962-01-02,Date present in supplied ohlc calendar,medium
...,...,...,...,...,...
5131,WMT,extreme_return,2022-07-26,"Simple raw-close return -0.0761191; review, no...",medium
5132,WMT,extreme_return,2022-11-15,"Simple raw-close return 0.0654672; review, not...",medium
5133,WMT,extreme_return,2023-11-16,"Simple raw-close return -0.080933; review, not...",medium
5134,WMT,extreme_return,2024-05-16,"Simple raw-close return 0.0698646; review, not...",medium


Post-repair gaps and return/zero-volume flags can remain; invalid retained prices cannot.


before                                \
                                  ts           open           high   
count                         397711  397711.000000  397711.000000   
mean   1998-04-17 21:18:45.292486144      46.285479      46.747660   
min              1962-01-02 00:00:00       0.000000       0.005208   
25%              1985-05-28 00:00:00       4.209938       4.240000   
50%              2000-01-06 00:00:00      19.850000      20.060000   
75%              2012-08-21 00:00:00      57.530000      58.130000   
max              2024-12-06 00:00:00     604.260000     608.630000   
std                              NaN      67.132594      67.778210   

                                                                  \
                 low          close      adj_close        volume   
count  397711.000000  397711.000000  377856.000000  3.977110e+05   
mean       45.816279      46.292903      51.652250  3.097773e+07   
min         0.004801       0.005208       0.002945  0.000000e+00   
25%         4.160000       4.200000       1.390000  1.730700e+06   
50%        19.630000      19.860000      11.960000  4.648200e+06   
75%        56.950000      57.530000      43.770000  1.102380e+07   
max       597.880000     605.400000   10442.510783  9.276606e+09   
std        66.477178      67.141879     194.584045  1.266237e+08   

                               after                                \
                                  ts           open           high   
count                         368246  368246.000000  368246.000000   
mean   1998-04-19 09:16:04.487869696      46.309394      46.771824   
min              1962-01-02 00:00:00       0.005208       0.005208   
25%              1985-05-28 00:00:00       4.210000       4.250000   
50%              2000-01-07 00:00:00      19.890000      20.090000   
75%              2012-08-21 00:00:00      57.560000      58.160000   
max              2024-12-06 00:00:00     604.260000     608.630000   
std                              NaN      67.143254      67.789319   

                                                                  \
                 low          close      adj_close        volume   
count  368246.000000  368246.000000  368246.000000  3.682460e+05   
mean       45.840375      46.317325      37.274670  3.095184e+07   
min         0.004801       0.005208       0.002945  0.000000e+00   
25%         4.170000       4.210000       1.310000  1.731514e+06   
50%        19.670000      19.910000      11.460000  4.649648e+06   
75%        56.997500      57.570000      41.570000  1.102160e+07   
max       597.880000     605.400000     605.400000  9.276606e+09   
std        66.489450      67.154939      63.587872  1.266153e+08   

                                        
          source_row adjustment_factor  
count  368246.000000     368246.000000  
mean   198781.151103          0.610934  
min         0.000000          0.101756  
25%     99282.250000          0.411819  
50%    198783.500000          0.622215  
75%    298200.750000          0.809411  
max    397710.000000          1.000000  
std    114822.872069          0.247345

## Injected edge cases
These deterministic examples exercise required cases even if the supplied database contains none. They are tests, not observations about the course data.

In [3]:
from checks import check_cleaning, check_metrics
check_cleaning()
check_metrics()
print("Injected repair, quarantine, gap, outlier and risk-metric checks passed.")

Injected repair, quarantine, gap, outlier and risk-metric checks passed.


In [4]:
save_cleaned(cleaned, db_path)
with sqlite3.connect(db_path) as connection:
    saved = pd.read_sql_query("SELECT * FROM ohlc_hw_cleaned", connection)
    assert len(saved) == len(cleaned)
    assert len(pd.read_sql_query("SELECT * FROM ohlc_hw", connection)) == len(raw)
print(f"Saved {len(saved)} validated rows to {db_path}:ohlc_hw_cleaned")
# Local audit files make all removed/quarantined rows and both diagnoses reviewable.
audit = Path("data/audit")
audit.mkdir(parents=True, exist_ok=True)
for name, frame in {"quality_before": before, "quality_after": after, "removed": removed,
                    "quarantine": quarantine}.items():
    frame.to_csv(audit / f"{name}.csv", index=False)

Saved 368246 validated rows to data/data.db:ohlc_hw_cleaned
